# 01 — FCC Data Discovery

## Objective

Identify and profile the FCC Fiber to the Premises dataset used in the Vermont Broadband Access & Digital Divide Analysis.


> **Scope note (added during review).** This notebook profiles only the **Fiber to the Premises** file (technology code 50).
> The `50` in the file name `bdc_50_...` is Vermont's state FIPS code, not a technology code.
> All counts below (22 providers, 187,037 locations, ~91% of locations with a single provider) are **fiber-only** figures
> and must not be read as the whole Vermont broadband market. The all-technology analysis is in notebooks 02 and 03.

## 1. Data Sources


In [1]:
import sys
import platform
import pandas as pd

sources = pd.DataFrame(
    [
        {
            "source": "FCC Broadband Data Collection",
            "organization": "Federal Communications Commission",
            "dataset_role": "Broadband availability",
        },
        {
            "source": "Vermont PSD / VCGI",
            "organization": "State of Vermont",
            "dataset_role": "Historical broadband availability and infrastructure",
        },
        {
            "source": "American Community Survey",
            "organization": "U.S. Census Bureau",
            "dataset_role": "Broadband adoption and socioeconomic characteristics",
        },
    ]
)

sources


,source,organization,dataset_role
0,FCC Broadband Data Collection,Federal Communications Commission,Broadband availability
1,Vermont PSD / VCGI,State of Vermont,Historical broadband availability and infrastr...
2,American Community Survey,U.S. Census Bureau,Broadband adoption and socioeconomic character...


## 2. Availability vs. Adoption

**Availability** measures whether broadband service is reported as available.  
**Adoption** measures whether households actually subscribe to broadband service.
Availability should not be interpreted as adoption.


## 3. Load FCC Fiber Data


In [2]:
from pathlib import Path
import pandas as pd

FCC_FIBER_PATH = Path(
    "../data/raw/fcc/2025-12-31/"
    "bdc_50_FibertothePremises_fixed_broadband_D25_18aug2026.csv"
)

fiber = pd.read_csv(FCC_FIBER_PATH)

fiber.shape


(203946, 12)

## 4. Inspect Dataset Schema


In [3]:
fiber.info()


<class 'pandas.DataFrame'>
RangeIndex: 203946 entries, 0 to 203945
Data columns (total 12 columns):
 #   Column                         Non-Null Count   Dtype
---  ------                         --------------   -----
 0   frn                            203946 non-null  int64
 1   provider_id                    203946 non-null  int64
 2   brand_name                     203946 non-null  str  
 3   location_id                    203946 non-null  int64
 4   technology                     203946 non-null  int64
 5   max_advertised_download_speed  203946 non-null  int64
 6   max_advertised_upload_speed    203946 non-null  int64
 7   low_latency                    203946 non-null  int64
 8   business_residential_code      203946 non-null  str  
 9   state_usps                     203946 non-null  str  
 10  block_geoid                    203946 non-null  int64
 11  h3_res8_id                     203946 non-null  str  
dtypes: int64(8), str(4)
memory usage: 24.9 MB


In [4]:
# location_id is part of the FCC Fabric (CostQuest IP): never displayed or published
fiber.drop(columns=["location_id"]).head()

,frn,provider_id,brand_name,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid,h3_res8_id
0,3768165,130317,Xfinity,50,100,100,1,B,VT,500110102002027,882bab5303fffff
1,3768165,130317,Xfinity,50,200,200,1,B,VT,500110109003016,882bab481bfffff
2,3768165,130317,Xfinity,50,1000,1000,1,B,VT,500110109003024,882bab48cdfffff
3,3768165,130317,Xfinity,50,100,100,1,B,VT,500110103001010,882babc865fffff
4,3768165,130317,Xfinity,50,100,100,1,B,VT,500110103002051,882babc831fffff


## 5. Validate Technology


In [5]:
fiber["technology"].value_counts(dropna=False)


technology
50    203946
Name: count, dtype: int64

## 6. Validate State Coverage


In [6]:
fiber["state_usps"].value_counts(dropna=False)


state_usps
VT    203946
Name: count, dtype: int64

## 7. Inspect Location Type


In [7]:
fiber["business_residential_code"].value_counts(dropna=False)


business_residential_code
X    107103
R     91910
B      4933
Name: count, dtype: int64

## 8. Count Providers and Brands


In [8]:
fiber["provider_id"].nunique()


22

In [9]:
fiber["brand_name"].nunique()


22

In [10]:
fiber["brand_name"].value_counts().head(20)


brand_name
Fidium Fiber                  95107
ECFiber                       25000
Burlington Telecom            18050
Vermont Telephone Company     16171
GREEN MOUNTAIN ACCESS         12452
NEK Broadband                  9852
TDS Telecom                    6013
Maple Broadband                3348
GoNetspeed                     3288
Stowe Communications LLC       2962
DVFiber                        2700
Mansfield Community Fiber      1691
Xfinity                        1658
Kingdom Fiber                  1619
Topsham Communications         1000
Franklin Telephone Company      902
Topsham Telephone Company       860
FirstLight Fiber                573
Spectrum                        500
Trans-Video                     155
Name: count, dtype: int64

## 9. Count Unique Locations


In [11]:
fiber["location_id"].nunique()


187037

## 10. Measure Provider Availability Overlap


In [12]:
providers_per_location = (
    fiber.groupby("location_id")["provider_id"]
    .nunique()
)

providers_per_location.describe()


count    187037.000000
mean          1.090405
std           0.291128
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           3.000000
Name: provider_id, dtype: float64

In [13]:
providers_per_location.value_counts().sort_index()


provider_id
1    170364
2     16437
3       236
Name: count, dtype: int64

## 11. Inspect Advertised Speeds


In [14]:
fiber[
    [
        "max_advertised_download_speed",
        "max_advertised_upload_speed",
    ]
].describe()


,max_advertised_download_speed,max_advertised_upload_speed
count,203946.000000,203946.000000
mean,2438.096604,2430.842684
std,2628.840363,2633.543934
min,10.000000,1.000000
25%,1000.000000,1000.000000
50%,2000.000000,2000.000000
75%,2000.000000,2000.000000
max,100000.000000,100000.000000


In [15]:
fiber[
    [
        "max_advertised_download_speed",
        "max_advertised_upload_speed",
    ]
].value_counts().head(20)


max_advertised_download_speed  max_advertised_upload_speed
2000                           2000                           95837
1000                           1000                           78698
10000                          10000                          17490
8000                           8000                            6055
300                            300                             1890
1000                           500                             1811
200                            200                              970
1200                           35                               417
100                            100                              268
50                             50                               146
150                            150                               97
10                             1                                 67
500                            500                               63
2000                           250                       

## 12. Check Duplicate Records


In [16]:
fiber.duplicated().sum()


np.int64(0)

In [17]:
fiber[
    [
        "provider_id",
        "location_id",
        "technology",
        "business_residential_code",
    ]
].duplicated().sum()


np.int64(0)

In [18]:
candidate_key_duplicates = fiber.duplicated(
    subset=["location_id", "provider_id", "technology"]
).sum()

print(
    "Duplicate location-provider-technology combinations:",
    candidate_key_duplicates,
)


Duplicate location-provider-technology combinations: 0


## 13. Summarize Provider Coverage


In [19]:
provider_summary = (
    fiber.groupby(["provider_id", "brand_name"])
    .agg(
        records=("location_id", "size"),
        unique_locations=("location_id", "nunique"),
    )
    .sort_values("unique_locations", ascending=False)
)

provider_summary


,,records,unique_locations
provider_id,brand_name,,
130335,Fidium Fiber,95107,95107
440199,ECFiber,25000,25000
131167,Burlington Telecom,18050,18050
131426,Vermont Telephone Company,16171,16171
131175,GREEN MOUNTAIN ACCESS,11289,11289
450083,NEK Broadband,9852,9852
131310,TDS Telecom,6013,6013
480002,Maple Broadband,3348,3348
131378,GoNetspeed,3288,3288


## 14. Validate Provider–Brand Relationships


In [20]:
provider_brand_check = (
    fiber.groupby("provider_id")["brand_name"]
    .nunique()
    .sort_values(ascending=False)
)

provider_brand_check


provider_id
130269    2
130235    1
510051    1
480002    1
470119    1
450083    1
440199    1
420059    1
400036    1
390166    1
240090    1
190405    1
131426    1
131378    1
131310    1
131175    1
131167    1
130993    1
130514    1
130335    1
130317    1
530001    1
Name: brand_name, dtype: int64

In [21]:
provider_brand_relationship = (
    fiber.groupby("provider_id")
    .agg(
        brand_count=("brand_name", "nunique"),
        brands=("brand_name", lambda x: sorted(x.unique())),
    )
    .query("brand_count > 1")
)

provider_brand_relationship


,brand_count,brands
provider_id,,
130269,2,"[Topsham Communications, Topsham Telephone Com..."


In [22]:
brand_provider_relationship = (
    fiber.groupby("brand_name")
    .agg(
        provider_count=("provider_id", "nunique"),
        provider_ids=("provider_id", lambda x: sorted(x.unique())),
    )
    .query("provider_count > 1")
)

brand_provider_relationship


,provider_count,provider_ids
brand_name,,
GREEN MOUNTAIN ACCESS,2,"[131175, 530001]"


## 15. Define Residential Analysis Population


In [23]:
fiber_residential = fiber[
    fiber["business_residential_code"].isin(["R", "X"])
].copy()

print("Residential and mixed-use records:", len(fiber_residential))
print(
    "Unique residential and mixed-use locations:",
    fiber_residential["location_id"].nunique(),
)


Residential and mixed-use records: 199013
Unique residential and mixed-use locations: 183350


## 16. Measure Residential Provider Availability Overlap



In [24]:
residential_providers_per_location = (
    fiber_residential.groupby("location_id")["provider_id"]
    .nunique()
)

residential_providers_per_location.value_counts().sort_index()


provider_id
1    167877
2     15283
3       190
Name: count, dtype: int64

In [25]:
residential_providers_per_location.describe()


count    183350.000000
mean          1.085427
std           0.283200
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           3.000000
Name: provider_id, dtype: float64

## 17. Inspect Provider Records per Location


In [26]:
fiber.groupby(
    ["location_id", "provider_id"]
).size().value_counts().sort_index()


1    203946
Name: count, dtype: int64

## 18. Missing values / nulls

In [27]:
fiber.isna().sum().sort_values(ascending=False)

frn                              0
provider_id                      0
brand_name                       0
location_id                      0
technology                       0
max_advertised_download_speed    0
max_advertised_upload_speed      0
low_latency                      0
business_residential_code        0
state_usps                       0
block_geoid                      0
h3_res8_id                       0
dtype: int64

## 19. Validación de low_latency

In [28]:
fiber["low_latency"].value_counts(dropna=False)

low_latency
1    203812
0       134
Name: count, dtype: int64

## 20. Validación del block_geoid

In [29]:
print(fiber["block_geoid"].astype(str).str.len().value_counts())
print(fiber["block_geoid"].astype(str).str.startswith("50").value_counts())

block_geoid
15    203946
Name: count, dtype: int64
block_geoid
True    203946
Name: count, dtype: int64


## 21. Data Discovery Findings


- The dataset contains 203,946 Fiber-to-the-Premises availability records.
- These records cover 187,037 unique FCC broadband-serviceable locations.
- 183,350 unique locations have residential or mixed residential/business fiber availability.
- 22 fiber provider IDs and 22 brand names are present (20 providers serve residential/mixed locations).
- Provider availability overlaps at some locations, with up to 3 fiber providers reported at a single location.
- Among residential/mixed fiber locations, 167,877 of 183,350 (91.6%) have exactly one fiber provider. This describes fiber competition only.
- The practical grain of this Fiber dataset is one record per location_id + provider_id + technology.
- provider_id and brand_name do not have a guaranteed one-to-one relationship.
- No fully duplicated rows were identified.